In [ ]:
import os
import glob
import xarray as xr
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import seaborn as sns

In [ ]:
# surface dataset created from Sean's script
surdat = xr.open_dataset(os.path.join('/glade/work/afoster/ABoVE/for_adrianna',
                                      'surfdata_0.9x1.25_hist_2000_16pfts_c251215.nc'))

In [ ]:
# observational data - basal area by pft
top_dir = '/glade/work/afoster/ABoVE'
fig_dir = os.path.join(top_dir, 'figures')
obs_dir = os.path.join(top_dir, 'obs_data')
site_dat = pd.read_csv(os.path.join(obs_dir, 'site_dat_fixed.csv'))
obs_dat = pd.read_csv(os.path.join(obs_dir, 'species_dat.csv'))
mega_sites = pd.read_csv(os.path.join(obs_dir, 'compiled_sites_1deg.csv'))

In [ ]:
# pfts we want to use
pfts = ['white_spruce', 'black_spruce', 'larch', 'deciduous_tree',
        'deciduous_shrub', 'evergreen_shrub', 'jack_pine', 'juniper']

In [ ]:
# points from observations
site_info = mega_sites[['mega_site', 'latitude', 'longitude']].drop_duplicates()
points = [(p1, p2) for p1, p2 in zip(site_info['latitude'], site_info['longitude'])]

In [ ]:
# surface data points
points_surf = [(p1, p2) for p1, p2 in zip(surdat.LATIXY.values, surdat.LONGXY.values)]
surfpoints_array = np.array(points_surf, dtype=object)

In [ ]:
gridcells = []
for point in points:
    matching_point = find_closest_point(point, surdat.LATIXY.values,
                                        surdat.LONGXY.values)
    gridcells.append(np.where(np.all(surfpoints_array == matching_point, axis=1))[0][0])
site_info['gridcell'] = gridcells

In [ ]:
surf_data_mod = surdat.copy(deep=True)
pct_nat_pft = surf_data_mod.PCT_NAT_PFT

sites = np.unique(site_info['mega_site'])
for i, site in enumerate(sites):
    sub_df = mega_sites[mega_sites.mega_site == site].copy()
    sub_info = site_info[site_info.mega_site == site].copy()
    grid = sub_info.gridcell.values[0]
    non_veg = pct_nat_pft[0, grid].values
    basum = sub_df['prop_BA'].sum() + non_veg
    sub_df.loc[:, 'prop_BA_nv'] = (sub_df['prop_BA']) / basum * 100.0

    for j in range(len(pct_nat_pft)):
            if j > 0:
                if j < 9:
                    sub = sub_df[sub_df['nwt.pft'] == pfts[j-1]]
                    if len(sub) > 0:
                        surf_data_mod["PCT_NAT_PFT"][j, grid] = sub.prop_BA_nv.values[0]
                    else:
                        surf_data_mod["PCT_NAT_PFT"][j, grid] = 0.0
                else:
                    surf_data_mod["PCT_NAT_PFT"][j, grid] = 0.0
    pft_pct = surf_data_mod["PCT_NAT_PFT"][:, grid]
    total = pft_pct.sum(dim="natpft")
    pft_norm = xr.where(
            total > 0,
            100.0 * pft_pct / total,
            0.0
        )
    surf_data_mod["PCT_NAT_PFT"][:, grid] = pft_norm

In [ ]:
surf_data_mod.to_netcdf(path='/glade/work/afoster/ABoVE/for_adrianna/surfdata_0.9x1.25_hist_2000_16pfts_c251215_updated.nc',
                        mode="w", format="NETCDF3_64BIT")

In [ ]:
surf_data_mod = xr.open_dataset('/glade/work/afoster/ABoVE/for_adrianna/surfdata_0.9x1.25_hist_2000_16pfts_c251215_updated.nc')

In [ ]:
surf_pfts = ['non-veg', 'white_spruce', 'black_spruce', 'larch', 'deciduous_tree',
              'deciduous_shrub', 'evergreen_shrub', 'jack_pine', 'juniper']
pft_dict = {}
for i, pft in enumerate(surf_pfts):
    pft_dict[pft] = surf_data_mod.PCT_NAT_PFT.isel(natpft=i).values

In [ ]:
pft_df = pd.DataFrame(pft_dict)
pft_df['gridcell'] = pft_df.index
pft_df = pft_df.merge(site_info, on='gridcell')
pft_df = pft_df.melt(id_vars=['gridcell', 'mega_site', 'latitude', 'longitude'],
                     var_name='pft', value_name='percent')
pft_df = pft_df.sort_values(by='pft')

In [ ]:
df_pivot = pft_df.pivot(index='gridcell', columns='pft', values='percent')
df_pivot = df_pivot[surf_pfts]
df_pivot.to_csv(os.path.join(obs_dir, 'surdat_composition.csv'))

In [ ]:
ax = df_pivot.plot(kind='barh', stacked=True, figsize=(10, 6),
                   color=[color_map[col] for col in df_pivot.columns])
plt.ylabel('Grid Cell')
plt.xlabel('Percent (%)')
plt.title('PFT Composition')
plt.legend(title='PFT', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'pft_composition.png'))